# 浙江水利 · 全省实时水情爬虫

这个 Notebook 保留原来的使用方式：使用 Selenium 打开浙江省水利厅实时水情页面，再用 BeautifulSoup 解析页面里的 `div.line` / `div.cell` 表格。没有改成 `.py` 脚本，也没有切换到 JSON 接口版。

推荐运行顺序：

1. 先运行“配置与依赖”单元格。
2. 调试时设置 `USE_TEST_CITIES = True`、`TEST_CITIES = ["杭州市"]`、`MAX_WORKERS = 1`、`RUN_ONCE = True`、`HEADLESS = False`。
3. 正式全省采集时设置 `USE_TEST_CITIES = False`、`MAX_WORKERS = 3`、`RUN_ONCE = False`、`HEADLESS = True`、`INTERVAL = 1800`。
4. 运行最后的主采集单元格 `run_crawler()`。

依赖安装示例：

```bash
pip install selenium beautifulsoup4
```

ChromeDriver 说明：

- 本 Notebook 会使用 `webdriver-manager` 自动安装与当前 Chrome 匹配的 ChromeDriver；失败时回退到 Selenium Manager。
- 如果提示找不到 ChromeDriver，请确认本机 Chrome 浏览器版本，并安装匹配的 ChromeDriver。
- 调试时可把 `HEADLESS = False`，观察浏览器是否正常打开页面。

常见问题：

- 页面无数据：检查网络、城市参数，或查看 `zhejiang_water_data/debug` 中保存的 HTML / 截图。
- ChromeDriver 异常：确认 ChromeDriver 路径和浏览器版本匹配。
- 数据库被锁定：关闭正在打开该数据库的其他程序，稍后重试。
- 长期运行：设置 `RUN_ONCE = False`，程序会按 `INTERVAL` 秒循环采集。


Added rainfall collection: rainfall is written to `rainfall_records_v1`; water level remains in `water_level_records_v2`.


In [ ]:
import os
import re
import time
import json
import queue
import shutil
import sqlite3
import logging
import threading
import concurrent.futures
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Optional
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from bs4 import BeautifulSoup

# Selenium 仍然是本 Notebook 的主爬取方式。
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ===================== 配置 =====================

API_BASE = "https://sqfb.slt.zj.gov.cn:30050"
API_PATH = "/nuxtsyq/new/realtimeWater"

RUNTIME_ROOT = Path(r"E:\\AAAqian\\storm_surge_runtime_data")
DATA_PATH = RUNTIME_ROOT / "zhejiang_water_data"
DB_FILE = "zhejiang_water_data.db"
LOG_FILE = "zhejiang_crawler.log"
DEBUG_DIR = "debug"

CITIES = [
    "杭州市", "宁波市", "温州市", "嘉兴市", "湖州市",
    "绍兴市", "金华市", "衢州市", "舟山市", "台州市",
    "丽水市",
]

USE_TEST_CITIES = False

# TEST_CITIES = ["杭州市"]  # 调试时使用
# USE_TEST_CITIES = True   # True 只爬 TEST_CITIES，False 爬全部 CITIES

MAX_WORKERS = 3
INTERVAL = 300
MAX_RETRIES = 5
RETRY_DELAY = 10
RUN_ONCE = False
HEADLESS = True
CHROME_BINARY_PATH = r"C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"
SAVE_DEBUG_HTML = True
SAVE_DEBUG_SCREENSHOT = True
CREATE_DB_BACKUP = False  # True 时启动前生成 .bak；默认关闭，避免备份文件堆积
PAGE_WAIT_TIMEOUT = 20
AREA_TYPE = "全省"
NEW_TABLE = "water_level_records_v2"
RAIN_TABLE = "rainfall_records_v1"
COLLECT_RAINFALL = True
RAIN_API_BASE = "https://sqfb.slt.zj.gov.cn"
RAIN_API_PATH = "/rest/newList/getNewTotalRainList"
RAIN_LOOKBACK_HOURS = 1
RAIN_REQUEST_TIMEOUT = 60

# 如需把旧表 zhejiang_water_data 里的历史数据复制进 v2 表，可改成 True。
# 默认 False，避免对旧数据做额外写入；导出 Notebook 会在 v2 无数据时回退使用旧表。
MIGRATE_LEGACY_ON_INIT = False

DB_PATH = DATA_PATH / DB_FILE
LOG_PATH = DATA_PATH / "logs" / LOG_FILE
DEBUG_PATH = DATA_PATH / DEBUG_DIR

DATA_PATH.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
DEBUG_PATH.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("zhejiang_water_crawler")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.INFO)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)

print_lock = threading.Lock()


In [ ]:
def active_cities() -> List[str]:
    """根据配置返回本轮要采集的城市。"""
    if USE_TEST_CITIES:
        return [city for city in TEST_CITIES if city in CITIES]
    return list(CITIES)


def safe_float(value: str) -> Optional[float]:
    """安全转换水位字符串，转换失败返回 None。"""
    if value is None:
        return None
    text = str(value).strip().replace(",", "")
    if not text or text in {"--", "-", "—", "暂无"}:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def parse_reported_at(time_str: str, now: Optional[datetime] = None) -> Optional[str]:
    """把页面中的上报时间尽量补成年月日时分秒。"""
    if not time_str:
        return None
    now = now or datetime.now()
    text = time_str.strip()
    formats = [
        ("%Y-%m-%d %H:%M:%S", False),
        ("%Y-%m-%d %H:%M", False),
        ("%m-%d %H:%M:%S", True),
        ("%m-%d %H:%M", True),
    ]
    for fmt, needs_year in formats:
        try:
            parsed = datetime.strptime(text, fmt)
            if needs_year:
                parsed = parsed.replace(year=now.year)
                if parsed - now > timedelta(days=7):
                    parsed = parsed.replace(year=now.year - 1)
            return parsed.strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            continue
    logger.debug("无法解析上报时间: %s", time_str)
    return None


def extract_station_code(name_cell, station_name: str) -> str:
    """从站名单元格 title 中提取站码，兼容大小写字母、数字、短横线和下划线。"""
    title = (name_cell.get("title") or "").strip()
    if not title:
        return ""

    patterns = [
        r"(?:站码|测站编码|编码|code|Code)[:：\s]*([A-Za-z0-9_-]+)",
        r"([A-Za-z0-9_-]+)$",
    ]
    for pattern in patterns:
        match = re.search(pattern, title)
        if match:
            code = match.group(1).strip()
            if code and code != station_name:
                return code
    return ""


def build_city_url(city: str) -> str:
    params = {
        "areaFlag": "1",
        "sss": city,
        "ssx": "",
        "zl": "ZZ,ZQ,DD,TT,",
        "sklx": "",
        "ly": "",
        "sfcj": "0",
        "bxdj": "1,2,3,4,5",
        "zm": "",
        "cjly": "",
        "bx": "0",
    }
    return f"{API_BASE}{API_PATH}?{urlencode(params)}"


def sanitize_filename(value: str) -> str:
    return re.sub(r"[\\/:*?\"<>|\s]+", "_", value or "unknown").strip("_")


def save_debug_page(driver, city: str, attempt: int, reason: str) -> None:
    """保存当前 HTML 和截图，方便排查页面结构变化。"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{timestamp}_{sanitize_filename(city)}_attempt{attempt}_{sanitize_filename(reason)}"

    if SAVE_DEBUG_HTML:
        html_path = DEBUG_PATH / f"{prefix}.html"
        html_path.write_text(driver.page_source or "", encoding="utf-8")
        logger.debug("debug HTML 已保存: %s", html_path)

    if SAVE_DEBUG_SCREENSHOT:
        screenshot_path = DEBUG_PATH / f"{prefix}.png"
        try:
            driver.save_screenshot(str(screenshot_path))
            logger.debug("debug 截图已保存: %s", screenshot_path)
        except Exception as exc:
            logger.debug("保存 debug 截图失败: %s", exc)


In [ ]:
class DriverPool:
    """为并发城市采集提供少量 Chrome 实例，避免一次启动过多浏览器。"""

    def __init__(self, size: int):
        self.size = max(1, int(size))
        self._drivers = []
        self._available = queue.Queue()

    def initialize(self) -> None:
        logger.info("正在创建 %s 个 Chrome 实例", self.size)
        for _ in range(self.size):
            driver = self._create_driver()
            self._drivers.append(driver)
            self._available.put(driver)
        logger.info("Chrome 实例已就绪")

    def _create_driver(self):
        options = Options()
        if HEADLESS:
            options.add_argument("--headless=new")
        if CHROME_BINARY_PATH and Path(CHROME_BINARY_PATH).exists():
            options.binary_location = CHROME_BINARY_PATH
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--lang=zh-CN")

        try:
            chromedriver_path = ChromeDriverManager().install()
            service = Service(chromedriver_path)
            logger.info("使用 webdriver-manager ChromeDriver: %s", chromedriver_path)
            return webdriver.Chrome(service=service, options=options)
        except Exception as exc:
            logger.warning("webdriver-manager 创建 ChromeDriver 失败，改用 Selenium Manager: %s", exc)
            return webdriver.Chrome(options=options)

    def acquire(self):
        return self._available.get()

    def release(self, driver) -> None:
        if driver is not None:
            self._available.put(driver)

    def close_all(self) -> None:
        logger.info("正在关闭 Chrome 实例")
        while not self._available.empty():
            try:
                self._available.get_nowait()
            except queue.Empty:
                break
        for driver in self._drivers:
            try:
                driver.quit()
            except Exception as exc:
                logger.debug("关闭 Chrome 实例时发生异常: %s", exc)
        self._drivers.clear()


def wait_for_water_table(driver) -> None:
    """等待表格结构、数据行或“暂无数据”提示出现。"""
    wait = WebDriverWait(driver, PAGE_WAIT_TIMEOUT)
    wait.until(
        lambda d: d.find_elements(By.CSS_SELECTOR, "div.line")
        or "暂无数据" in d.page_source
        or "水位" in d.page_source
    )


In [ ]:
def parse_water_rows(html: str, city: str, source_url: str, crawl_now: datetime) -> List[Dict]:
    """解析页面表格，保留 div.line / div.cell 的原始解析思路。"""
    soup = BeautifulSoup(html, "html.parser")
    lines = soup.find_all("div", class_="line")
    records: List[Dict] = []
    current_station_type = ""
    crawl_time = crawl_now.isoformat(timespec="seconds")

    for line_index, line in enumerate(lines):
        sub = line.find("span", class_="sub")
        if sub:
            current_station_type = sub.get_text(strip=True)
            continue

        cells = line.find_all("div", class_="cell")
        if len(cells) < 5:
            logger.debug("%s 第 %s 行 cell 数量不足: %s", city, line_index, len(cells))
            continue

        order_no = cells[0].get_text(strip=True)
        city_county = cells[1].get_text(strip=True)
        station_name = cells[2].get_text(strip=True)
        time_str = cells[3].get_text(strip=True)
        raw_water_level = cells[4].get_text(strip=True)

        if order_no == "序" or station_name == "站名":
            continue
        if not station_name or not raw_water_level:
            logger.debug("%s 第 %s 行缺少站名或水位", city, line_index)
            continue

        water_level = safe_float(raw_water_level)
        if water_level is None:
            logger.debug("%s 第 %s 行水位无法转成数值: %s", city, line_index, raw_water_level)
            continue

        reported_at = parse_reported_at(time_str, crawl_now)
        if not reported_at:
            reported_at = f"unparsed:{time_str}"

        station_code = extract_station_code(cells[2], station_name)

        records.append({
            "city": city,
            "city_county": city_county,
            "station_name": station_name,
            "station_code": station_code,
            "station_type": current_station_type,
            "order_no": order_no,
            "time_str": time_str,
            "reported_at": reported_at,
            "water_level": water_level,
            "raw_water_level": raw_water_level,
            "area_type": AREA_TYPE,
            "crawl_time": crawl_time,
            "source_url": source_url,
        })

    return records


def fetch_city_data(driver_pool: DriverPool, city: str) -> List[Dict]:
    """采集单个城市，失败后有限重试，不让整个程序卡死。"""
    records: List[Dict] = []
    driver = driver_pool.acquire()
    url = build_city_url(city)
    logger.info("%s 请求 URL: %s", city, url)

    try:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                crawl_now = datetime.now()
                driver.get(url)
                wait_for_water_table(driver)
                html = driver.page_source or ""
                records = parse_water_rows(html, city, url, crawl_now)

                if records:
                    logger.info("%s 解析到 %s 条数据", city, len(records))
                    return records

                if "暂无数据" in html:
                    logger.info("%s 页面返回暂无数据", city)
                    save_debug_page(driver, city, attempt, "no_data")
                    return []

                logger.warning("%s 第 %s 次尝试未解析到数据", city, attempt)
                save_debug_page(driver, city, attempt, "empty_parse")
                raise ValueError("页面结构异常或未解析到数据")

            except Exception as exc:
                logger.warning("%s 第 %s/%s 次尝试失败: %s", city, attempt, MAX_RETRIES, exc)
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)
                else:
                    logger.error("%s 达到最大重试次数，返回空列表", city)
        return []
    finally:
        driver_pool.release(driver)



def build_rainfall_period(crawl_now: datetime) -> tuple:
    """Return the previous whole-hour rainfall period used by the realtime rain page."""
    period_end = crawl_now.replace(minute=0, second=0, microsecond=0)
    period_start = period_end - timedelta(hours=RAIN_LOOKBACK_HOURS)
    return period_start, period_end


def build_rainfall_url(period_start: datetime, period_end: datetime) -> str:
    params = {
        "areaFlag": "1",
        "sss": "\u5168\u90e8",
        "ssx": "",
        "st": period_start.strftime("%Y-%m-%dT%H:%M:%S"),
        "et": period_end.strftime("%Y-%m-%dT%H:%M:%S"),
        "ly": "",
        "max": "",
        "min": "0",
        "bool": "false",
        "bxdj": "1,2,3,4,5,",
        "zm": "",
        "type": "0",
        "lx": "QX,ME,SX,DS",
    }
    return f"{RAIN_API_BASE}{RAIN_API_PATH}?{urlencode(params)}"


def fetch_json_url(url: str, timeout: int = RAIN_REQUEST_TIMEOUT):
    request = Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Referer": "https://sqfb.slt.zj.gov.cn/weIndex.html",
            "Accept": "application/json, text/plain, */*",
        },
    )
    with urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8", errors="ignore"))


def parse_rainfall_groups(data: Dict, period_start: datetime, period_end: datetime, crawl_now: datetime, cities: List[str]) -> List[Dict]:
    """Flatten realtime rainfall groups into station rows."""
    records: List[Dict] = []
    seen = set()
    active_city_set = set(cities)
    filter_cities = active_city_set and active_city_set != set(CITIES)
    period_start_text = period_start.strftime("%Y-%m-%d %H:%M:%S")
    period_end_text = period_end.strftime("%Y-%m-%d %H:%M:%S")
    crawl_time = crawl_now.isoformat(timespec="seconds")

    for rain_range, rows in (data or {}).items():
        if rain_range == "top30" or not isinstance(rows, list):
            continue
        for item in rows:
            if not isinstance(item, dict):
                continue
            city = item.get("sss") or ""
            if filter_cities and city not in active_city_set:
                continue
            station_name = item.get("zm") or ""
            raw_rainfall = item.get("yl")
            rainfall = safe_float(raw_rainfall)
            if not station_name or rainfall is None:
                continue
            station_code = item.get("zh") or ""
            city_county = item.get("ssx") or ""
            dedup_key = (city, city_county, station_name, station_code, period_start_text, period_end_text)
            if dedup_key in seen:
                continue
            seen.add(dedup_key)
            records.append({
                "city": city,
                "city_county": city_county,
                "station_name": station_name,
                "station_code": station_code,
                "station_type": "rainfall",
                "period_start": period_start_text,
                "period_end": period_end_text,
                "rainfall": rainfall,
                "raw_rainfall": str(raw_rainfall),
                "rainfall_range": rain_range,
                "longitude": item.get("jd"),
                "latitude": item.get("wd"),
                "area_type": AREA_TYPE,
                "crawl_time": crawl_time,
            })
    return records


def fetch_rainfall_data(cities: List[str], crawl_now: Optional[datetime] = None) -> List[Dict]:
    """Collect station rainfall for the previous whole hour from the realtime rain endpoint."""
    crawl_now = crawl_now or datetime.now()
    period_start, period_end = build_rainfall_period(crawl_now)
    url = build_rainfall_url(period_start, period_end)
    logger.info("rainfall request URL: %s", url)
    try:
        payload = fetch_json_url(url)
        records = parse_rainfall_groups(payload, period_start, period_end, crawl_now, cities)
        logger.info("rainfall parsed %s records for %s to %s", len(records), period_start, period_end)
        return records
    except Exception as exc:
        logger.error("rainfall collection failed: %s", exc)
        return []


In [ ]:
def backup_database(db_path: Path) -> Optional[Path]:
    """修改数据库结构前复制一份 .bak，不删除旧数据库。"""
    if not db_path.exists():
        return None
    backup_dir = db_path.parent / "backups"
    backup_dir.mkdir(parents=True, exist_ok=True)
    backup_path = backup_dir / f"{db_path.name}.{datetime.now().strftime('%Y%m%d_%H%M%S')}.bak"
    shutil.copy2(db_path, backup_path)
    logger.info("数据库备份已创建: %s", backup_path)
    return backup_path


def ensure_database(db_path: Path = DB_PATH, make_backup: bool = False, allow_migrate: bool = True) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if make_backup:
        backup_database(db_path)

    conn = sqlite3.connect(db_path)
    try:
        cursor = conn.cursor()
        cursor.execute(f"""
            CREATE TABLE IF NOT EXISTS {NEW_TABLE} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                city TEXT NOT NULL DEFAULT '',
                city_county TEXT NOT NULL DEFAULT '',
                station_name TEXT NOT NULL DEFAULT '',
                station_code TEXT NOT NULL DEFAULT '',
                station_type TEXT,
                order_no TEXT,
                time_str TEXT,
                reported_at TEXT NOT NULL DEFAULT '',
                water_level REAL,
                raw_water_level TEXT,
                area_type TEXT,
                crawl_time TEXT,
                source_url TEXT,
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                UNIQUE(city, city_county, station_name, station_code, reported_at)
            )
        """)

        cursor.execute(f"""
            CREATE TABLE IF NOT EXISTS {RAIN_TABLE} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                city TEXT NOT NULL DEFAULT '',
                city_county TEXT NOT NULL DEFAULT '',
                station_name TEXT NOT NULL DEFAULT '',
                station_code TEXT NOT NULL DEFAULT '',
                station_type TEXT,
                period_start TEXT NOT NULL DEFAULT '',
                period_end TEXT NOT NULL DEFAULT '',
                rainfall REAL,
                raw_rainfall TEXT,
                rainfall_range TEXT,
                longitude REAL,
                latitude REAL,
                area_type TEXT,
                crawl_time TEXT,
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                UNIQUE(city, city_county, station_name, station_code, period_start, period_end)
            )
        """)
        conn.commit()
        if MIGRATE_LEGACY_ON_INIT and allow_migrate:
            migrate_legacy_rows(conn)
    finally:
        conn.close()


def legacy_table_exists(conn) -> bool:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='zhejiang_water_data'")
    return cursor.fetchone() is not None


def migrate_legacy_rows(conn) -> int:
    """可选：把旧表数据复制到 v2 表。只写新表，不删除旧数据。"""
    if not legacy_table_exists(conn):
        return 0

    cursor = conn.cursor()
    cursor.execute("SELECT * FROM zhejiang_water_data")
    rows = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    migrated = []
    now = datetime.now()

    for row in rows:
        item = dict(zip(columns, row))
        raw_level = item.get("raw_water_level") or item.get("water_level")
        water_level = safe_float(raw_level)
        if water_level is None:
            continue
        time_str = item.get("time_str") or ""
        reported_at = item.get("reported_at") or parse_reported_at(time_str, now) or f"unparsed:{time_str}"
        migrated.append({
            "city": item.get("city") or "",
            "city_county": item.get("city_county") or "",
            "station_name": item.get("station_name") or "",
            "station_code": item.get("station_code") or "",
            "station_type": item.get("station_type") or "",
            "order_no": item.get("order") or item.get("order_no") or "",
            "time_str": time_str,
            "reported_at": reported_at,
            "water_level": water_level,
            "raw_water_level": str(raw_level),
            "area_type": item.get("area_type") or AREA_TYPE,
            "crawl_time": item.get("crawl_time") or now.isoformat(timespec="seconds"),
            "source_url": "legacy:zhejiang_water_data",
        })

    return save_to_sqlite(migrated, make_backup=False)


def save_to_sqlite(data: List[Dict], db_path: Path = DB_PATH, make_backup: bool = False) -> int:
    if not data:
        logger.info("本轮没有可写入数据")
        return 0

    ensure_database(db_path, make_backup=make_backup, allow_migrate=False)
    conn = sqlite3.connect(db_path)
    try:
        cursor = conn.cursor()
        sql = f"""
            INSERT INTO {NEW_TABLE} (
                city, city_county, station_name, station_code, station_type, order_no,
                time_str, reported_at, water_level, raw_water_level, area_type, crawl_time, source_url
            ) VALUES (
                :city, :city_county, :station_name, :station_code, :station_type, :order_no,
                :time_str, :reported_at, :water_level, :raw_water_level, :area_type, :crawl_time, :source_url
            )
            ON CONFLICT(city, city_county, station_name, station_code, reported_at) DO UPDATE SET
                station_type = excluded.station_type,
                order_no = excluded.order_no,
                time_str = excluded.time_str,
                water_level = excluded.water_level,
                raw_water_level = excluded.raw_water_level,
                crawl_time = excluded.crawl_time,
                area_type = excluded.area_type,
                source_url = excluded.source_url,
                updated_at = CURRENT_TIMESTAMP
        """
        cursor.executemany(sql, data)
        conn.commit()
        logger.info("本轮写入/更新 %s 条数据到 %s", len(data), NEW_TABLE)
        return len(data)
    finally:
        conn.close()



def save_rainfall_to_sqlite(data: List[Dict], db_path: Path = DB_PATH) -> int:
    if not data:
        logger.info("no rainfall records to write")
        return 0

    ensure_database(db_path, make_backup=False, allow_migrate=False)
    conn = sqlite3.connect(db_path)
    try:
        cursor = conn.cursor()
        sql = f"""
            INSERT INTO {RAIN_TABLE} (
                city, city_county, station_name, station_code, station_type,
                period_start, period_end, rainfall, raw_rainfall, rainfall_range,
                longitude, latitude, area_type, crawl_time
            ) VALUES (
                :city, :city_county, :station_name, :station_code, :station_type,
                :period_start, :period_end, :rainfall, :raw_rainfall, :rainfall_range,
                :longitude, :latitude, :area_type, :crawl_time
            )
            ON CONFLICT(city, city_county, station_name, station_code, period_start, period_end) DO UPDATE SET
                station_type = excluded.station_type,
                rainfall = excluded.rainfall,
                raw_rainfall = excluded.raw_rainfall,
                rainfall_range = excluded.rainfall_range,
                longitude = excluded.longitude,
                latitude = excluded.latitude,
                area_type = excluded.area_type,
                crawl_time = excluded.crawl_time,
                updated_at = CURRENT_TIMESTAMP
        """
        cursor.executemany(sql, data)
        conn.commit()
        logger.info("wrote/updated %s rainfall records into %s", len(data), RAIN_TABLE)
        return len(data)
    finally:
        conn.close()


In [ ]:
def collect_once() -> int:
    """Run one collection round and write water level plus rainfall data."""
    cities = active_cities()
    if not cities:
        logger.warning("No cities configured for collection")
        return 0

    worker_count = max(1, min(MAX_WORKERS, len(cities)))
    logger.info("Water cities this round: %s", ", ".join(cities))
    logger.info("Chrome workers: %s", worker_count)

    driver_pool = DriverPool(worker_count)
    all_records: List[Dict] = []
    try:
        driver_pool.initialize()
        with concurrent.futures.ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = {executor.submit(fetch_city_data, driver_pool, city): city for city in cities}
            for future in concurrent.futures.as_completed(futures):
                city = futures[future]
                try:
                    records = future.result()
                    all_records.extend(records)
                except Exception as exc:
                    logger.error("%s water collection task failed: %s", city, exc)

        logger.info("Water records parsed this round: %s", len(all_records))
        water_count = save_to_sqlite(all_records)
    finally:
        driver_pool.close_all()

    rain_count = 0
    if COLLECT_RAINFALL:
        rainfall_records = fetch_rainfall_data(cities)
        rain_count = save_rainfall_to_sqlite(rainfall_records)

    return water_count + rain_count


def run_crawler() -> None:
    """Run once when RUN_ONCE=True, otherwise loop by INTERVAL seconds."""
    ensure_database(DB_PATH, make_backup=CREATE_DB_BACKUP)

    while True:
        round_start = datetime.now()
        logger.info("========== collection start: %s ==========", round_start.isoformat(timespec="seconds"))
        upsert_count = collect_once()
        round_end = datetime.now()
        logger.info(
            "========== collection end: %s, wrote/updated %s records, elapsed %.1f seconds ==========",
            round_end.isoformat(timespec="seconds"),
            upsert_count,
            (round_end - round_start).total_seconds(),
        )

        if RUN_ONCE:
            logger.info("RUN_ONCE=True, exiting after this round")
            break

        logger.info("Waiting %s seconds before next round", INTERVAL)
        time.sleep(INTERVAL)


In [ ]:
# 主采集单元格
# 调试单城市建议：USE_TEST_CITIES=True, TEST_CITIES=["杭州市"], MAX_WORKERS=1, RUN_ONCE=True, HEADLESS=False
# 正式全省采集建议：USE_TEST_CITIES=False, MAX_WORKERS=3, RUN_ONCE=False, HEADLESS=True, INTERVAL=300, COLLECT_RAINFALL=True
run_crawler()
